# PY400 — Web Crawling and Scraping

Web scraping is the process of **extracting data from websites** programmatically. It is how datasets are built from public web pages — product prices, news articles, job listings, weather data, etc.

## Important Libraries

| Library | Purpose | Best For |
|---|---|---|
| **`requests`** | HTTP client — fetch web pages | Making GET/POST requests |
| **`BeautifulSoup` (bs4)** | HTML/XML parser | Simple, static pages |
| **`lxml`** | Fast XML/HTML parser | Speed, XPath queries |
| **`Scrapy`** | Full crawling framework | Large-scale crawls, pipelines |
| **`Selenium`** | Browser automation | JavaScript-rendered pages |
| **`Playwright`** | Modern browser automation | JS-heavy SPAs (faster than Selenium) |
| **`httpx`** | Modern async HTTP client | Async scraping |

## Ethics and Legality
- Always check the website's **`robots.txt`** (e.g., `https://example.com/robots.txt`)
- Respect **rate limits** — add delays between requests
- Do not scrape **personal data** without consent (GDPR, privacy laws)
- Check the **Terms of Service** — some sites explicitly prohibit scraping
- Prefer **official APIs** when available — they are faster, more reliable, and legal

> **Perspective:** BeautifulSoup + requests handles 80% of scraping tasks. Scrapy handles 15% (large scale, complex pipelines). Selenium/Playwright handles the remaining 5% (JavaScript-rendered content). Start simple, escalate only when needed.

---
## 1. `requests` — Fetching Web Pages

In [ ]:
## Fetching a web page with requests

import requests

url = 'https://httpbin.org/html'    # A safe test page
response = requests.get(url)

print(f"Status Code : {response.status_code}")   # 200 = success
print(f"Content Type: {response.headers.get('Content-Type')}")
print(f"Content Size: {len(response.text)} chars")
print(f"\nFirst 300 chars:\n{response.text[:300]}")

# Common status codes:
# 200 = OK, 301 = Redirect, 403 = Forbidden, 404 = Not Found, 429 = Rate Limited

---
## 2. `BeautifulSoup` — Parsing HTML

In [ ]:
## Parsing HTML with BeautifulSoup

from bs4 import BeautifulSoup

# Sample HTML (simulating a fetched page)
html = """
<html>
<body>
  <h1>Product Catalog</h1>
  <div class="product">
    <h2>Laptop</h2>
    <span class="price">₹75,000</span>
    <p class="rating">Rating: 4.5/5</p>
  </div>
  <div class="product">
    <h2>Phone</h2>
    <span class="price">₹25,000</span>
    <p class="rating">Rating: 4.2/5</p>
  </div>
  <div class="product">
    <h2>Tablet</h2>
    <span class="price">₹35,000</span>
    <p class="rating">Rating: 4.0/5</p>
  </div>
</body>
</html>
"""

soup = BeautifulSoup(html, 'html.parser')

# Find single element
title = soup.find('h1')
print(f"Title: {title.text}")

# Find all products
products = soup.find_all('div', class_='product')
print(f"\nFound {len(products)} products:")

for product in products:
    name = product.find('h2').text
    price = product.find('span', class_='price').text
    rating = product.find('p', class_='rating').text
    print(f"  {name}: {price} — {rating}")

In [ ]:
## Extracting data into a Pandas DataFrame

import pandas as pd

data = []
for product in products:
    data.append({
        'name':   product.find('h2').text,
        'price':  product.find('span', class_='price').text,
        'rating': product.find('p', class_='rating').text
    })

df = pd.DataFrame(data)

# Clean the data — remove ₹ and commas, extract numeric rating
df['price_num'] = df['price'].str.replace('[₹,]', '', regex=True).astype(int)
df['rating_num'] = df['rating'].str.extract(r'(\d+\.\d+)').astype(float)

print(df)

# Best Practice: Always clean scraped data immediately. Web data is messy —
# extra whitespace, HTML entities, inconsistent formatting.

---
## 3. CSS Selectors vs `find()` — Two Ways to Navigate HTML

| Approach | Syntax | Example |
|---|---|---|
| `find()` / `find_all()` | Tag + attributes | `soup.find('div', class_='price')` |
| `select()` / `select_one()` | CSS selectors | `soup.select('div.price > span')` |

In [ ]:
## CSS selectors with BeautifulSoup

# Select all product names using CSS selectors
names = soup.select('div.product h2')
print("CSS selector (div.product h2):")
for n in names:
    print(f"  {n.text}")

# Select all prices
prices = soup.select('span.price')
print("\nCSS selector (span.price):")
for p in prices:
    print(f"  {p.text}")

# Tip: CSS selectors are often more concise and familiar to web developers.
# Use browser DevTools (Inspect Element) to find the right selector.

---
## 4. Scrapy vs BeautifulSoup — When to Use Which

| Feature | BeautifulSoup + requests | Scrapy |
|---|---|---|
| **Learning curve** | Low | Medium-High |
| **Setup** | `pip install requests beautifulsoup4` | Full project structure |
| **Async / Concurrent** | No (manual with `asyncio`) | Built-in |
| **Rate limiting** | Manual (`time.sleep()`) | Built-in middleware |
| **Data pipelines** | Manual | Built-in (Item Pipeline) |
| **JavaScript rendering** | No | No (need Splash/Playwright add-on) |
| **Best for** | Quick, small scrapes (< 1000 pages) | Large-scale production crawls |

### Other Notable Tools
- **`Playwright`** (Microsoft) — modern alternative to Selenium, supports Chromium/Firefox/WebKit, async by default. Increasingly preferred over Selenium.
- **`selectolax`** — fastest HTML parser in Python (Cython-based), 10x faster than BeautifulSoup. Use for high-volume parsing.
- **`pandas.read_html(url)`** — one-liner to extract HTML tables into DataFrames. Amazing for Wikipedia tables, financial data tables, etc.

> **Perspective:** Before writing a scraper, always check if the site offers an **API** or a **data download**. Scraping is fragile — any website redesign breaks your code. APIs are stable contracts.

In [ ]:
## Bonus: pd.read_html() — scrape HTML tables in one line

# This reads ALL <table> elements from a URL or HTML string
html_table = """
<table>
  <tr><th>City</th><th>Population</th><th>Area (km²)</th></tr>
  <tr><td>Mumbai</td><td>20,411,000</td><td>603</td></tr>
  <tr><td>Delhi</td><td>16,787,000</td><td>1,484</td></tr>
  <tr><td>Bangalore</td><td>8,443,000</td><td>741</td></tr>
</table>
"""

tables = pd.read_html(html_table)
print(f"Found {len(tables)} table(s)")
print(tables[0])

# For real URLs: tables = pd.read_html('https://en.wikipedia.org/wiki/...')